In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("../src")

from sentence_transformers import SentenceTransformer
from data_loader import load_candidates
from job_representation import build_job_description
from retrieval import Retriever
from ranking import Ranker

# Load model only once
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

retriever = Retriever(embedding_model)
ranker = Ranker(embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
from data_loader import load_candidates

candidates = load_candidates(
    "../data/candidates.jsonl",
    limit=100
)


In [4]:
from docx import Document
from job_representation import build_job_description

doc = Document("../India_Runs_Hackathon/job_description.docx")

jd_text = "\n".join(
    para.text
    for para in doc.paragraphs
)

job = build_job_description(jd_text)

In [22]:
ranker.build_requirement_embeddings(job)

ranker.build_experience_embeddings(candidates)

ranker.build_document_embeddings(
    candidates
)

In [7]:
retrieved = retriever.retrieve_candidates(job, top_k=100)

candidate_lookup = {
    c.candidate_id: c
    for c in candidates
}

retrieved_candidates = [
    candidate_lookup[cid]
    for cid, _ in retrieved
]

rankings = ranker.rank_candidates(
    job,
    retrieved_candidates
)

AssertionError: Build BM25 index first.

In [23]:
rankings = ranker.rank_candidates(
    job,
    candidates
)

retrieval systems         Weight=3.0 Best=0.156
ranking systems           Weight=3.0 Best=0.089
recommendation systems    Weight=3.0 Best=0.114
search systems            Weight=3.0 Best=0.233
embeddings                Weight=2.5 Best=0.047
vector databases          Weight=2.5 Best=0.286
python                    Weight=2.0 Best=0.199
llms                      Weight=1.5 Best=0.073
fine-tuning               Weight=1.0 Best=0.161
bm25                      Weight=2.0 Best=0.123

Final Evidence Score = 0.150
retrieval systems         Weight=3.0 Best=0.206
ranking systems           Weight=3.0 Best=0.196
recommendation systems    Weight=3.0 Best=0.150
search systems            Weight=3.0 Best=0.304
embeddings                Weight=2.5 Best=0.043
vector databases          Weight=2.5 Best=0.138
python                    Weight=2.0 Best=0.256
llms                      Weight=1.5 Best=0.159
fine-tuning               Weight=1.0 Best=0.122
bm25                      Weight=2.0 Best=0.132

Final Evi

In [24]:
for i, result in enumerate(rankings[:10], 1):
    c = result["candidate"]

    print(f"\n{'='*60}")
    print(f"Rank {i}: {c.candidate_id}")

    for req, info in c.matched_experiences.items():

        exp = info["experience"]

        if exp is None:
            continue

        print(f"{req:25} | {info['score']:.3f} | {exp.title}")


Rank 1: CAND_0000031
retrieval systems         | 0.361 | Search Engineer
ranking systems           | 0.417 | Recommendation Systems Engineer
recommendation systems    | 0.508 | Recommendation Systems Engineer
search systems            | 0.409 | Search Engineer
embeddings                | 0.170 | NLP Engineer
vector databases          | 0.246 | Search Engineer
python                    | 0.200 | NLP Engineer
llms                      | 0.182 | Recommendation Systems Engineer
fine-tuning               | 0.258 | Applied ML Engineer
bm25                      | 0.166 | Search Engineer

Rank 2: CAND_0000021
retrieval systems         | 0.206 | Project Manager
ranking systems           | 0.181 | Project Manager
recommendation systems    | 0.216 | Customer Support
search systems            | 0.290 | Project Manager
embeddings                | 0.091 | Customer Support
vector databases          | 0.113 | Project Manager
python                    | 0.239 | Project Manager
llms                    

In [26]:
from sentence_transformers import util
scores = []

for req_emb in job.requirement_embeddings:
    for candidate in candidates:
        for exp in candidate.experiences:

            if exp.evidence_embedding is None:
                continue

            scores.append(
                util.cos_sim(req_emb, exp.evidence_embedding).item()
            )

print("Min :", min(scores))
print("Mean:", sum(scores) / len(scores))
print("Max :", max(scores))

Min : -0.07441392540931702
Mean: 0.1285762618255356
Max : 0.508325457572937


In [27]:
titles = {}

for c in candidates:

    title = c.experiences[0].title

    titles[title] = titles.get(title, 0) + 1

for k, v in sorted(
    titles.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]:
    print(v, k)

10 Business Analyst
10 Mechanical Engineer
10 Frontend Engineer
8 Project Manager
7 Graphic Designer
6 Operations Manager
5 Accountant
5 Full Stack Developer
4 Customer Support
4 Marketing Manager
4 QA Engineer
4 HR Manager
4 DevOps Engineer
4 Java Developer
3 Civil Engineer
3 Cloud Engineer
2 Software Engineer
1 Backend Engineer
1 Data Engineer
1 Recommendation Systems Engineer


In [ ]:
candidate =candidates[1]

for exp in candidate.experiences:
    print("TITLE:", exp.title)
    print("EVIDENCE:")
    print(repr(exp.evidence_text))
    print("-" * 60)

TITLE: Operations Manager
EVIDENCE:
'Operations Manager.\nManaged a team of 8 support agents handling tier-1 and tier-2 tickets; owned the escalation process to engineering and the customer-feedback loop to product.\nBuilt out the support knowledge base and the agent training program.'
------------------------------------------------------------
TITLE: Operations Manager
EVIDENCE:
'Operations Manager.\n'
------------------------------------------------------------
TITLE: Marketing Manager
EVIDENCE:
'Marketing Manager.\n'
------------------------------------------------------------
TITLE: Marketing Manager
EVIDENCE:
'Marketing Manager.\nOwned brand identity (logo, visual system, typography), packaging design, and digital creative across web and social.'
------------------------------------------------------------


In [9]:
rankings = ranker.rank_candidates(job, candidates)

for rank, result in enumerate(rankings[:20], start=1):

    candidate = result["candidate"]

    print("=" * 100)
    print(f"Rank : {rank}")
    print(f"ID   : {candidate.candidate_id}")
    print(f"Final Score      : {result['final_score']:.3f}")
    print(f"Evidence Score   : {result['evidence_score']:.3f}")
    print(f"Skill Score      : {result['skill_score']:.3f}")
    print(f"Experience Score : {result['experience_score']:.3f}")
    print(f"Behavior Score   : {result['behavior_score']:.3f}")

    print("\nMatched Requirements")

    for requirement, info in candidate.matched_experiences.items():

        exp = info["experience"]

        if exp is None:
            continue

        print(f"\n{requirement} ({info['score']:.3f})")
        print(f"Role    : {exp.title}")
        print(f"Company : {exp.company}")
        print(f"Evidence: {exp.evidence_text}")

    print("\nReasoning:")
    print(result["reasoning"])

retrieval systems         Weight=3.0 Best=0.230
ranking systems           Weight=3.0 Best=0.115
recommendation systems    Weight=3.0 Best=0.126
search systems            Weight=3.0 Best=0.231
embeddings                Weight=2.5 Best=0.054
vector databases          Weight=2.5 Best=0.256
python                    Weight=2.0 Best=0.129
llms                      Weight=1.5 Best=0.074
fine-tuning               Weight=1.0 Best=0.227
bm25                      Weight=2.0 Best=0.087

Final Evidence Score = 0.155
retrieval systems         Weight=3.0 Best=0.210
ranking systems           Weight=3.0 Best=0.266
recommendation systems    Weight=3.0 Best=0.203
search systems            Weight=3.0 Best=0.315
embeddings                Weight=2.5 Best=0.060
vector databases          Weight=2.5 Best=0.200
python                    Weight=2.0 Best=0.326
llms                      Weight=1.5 Best=0.310
fine-tuning               Weight=1.0 Best=0.220
bm25                      Weight=2.0 Best=0.263

Final Evi

In [7]:
candidate = rankings[0]["candidate"]

for req, info in candidate.matched_experiences.items():
    print("=" * 50)
    print(req)
    print("Score:", round(info["score"], 3))

    if info["experience"]:
        print("Title:", info["experience"].title)
        print(info["experience"].description[:200])

retrieval systems
Score: 0.371
Title: Search Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
ranking systems
Score: 0.45
Title: Recommendation Systems Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
recommendation systems
Score: 0.53
Title: Recommendation Systems Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: content metadata, user behavior signals, and item enga
search systems
Score: 0.418
Title: Search Engineer
Trained and shipped multiple ranking models for our product's discovery feed using XGBoost and LightGBM. Designed features across three families: conte

In [12]:
print(candidates[0].retrieval_document)

Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.
Headline: Backend Engineer | SQL, Spark, Cloud
Skills: Tailwind, NLP, Image Classification, Fine-tuning LLMs, Weights & Biases, Speech Recognition, Photoshop, TTS, LoRA, Apache Beam, AWS, Flask, BentoML, Milvus, GANs, Statistical Modeling, GCP
Education: B.E. Computer Science
Company: Mindtree
Experience Title: Backend Engineer
Experience Description: Im

In [11]:
for result in rankings[:20]:
    print(
        result["candidate_id"],
        round(result["evidence_score"], 3)
    )

CAND_0000031 0.334
CAND_0000082 0.189
CAND_0000083 0.159
CAND_0000097 0.18
CAND_0000001 0.172
CAND_0000074 0.178
CAND_0000021 0.18
CAND_0000042 0.191
CAND_0000024 0.175
CAND_0000026 0.178
CAND_0000020 0.173
CAND_0000014 0.16
CAND_0000084 0.178
CAND_0000002 0.176
CAND_0000065 0.166
CAND_0000045 0.167
CAND_0000036 0.162
CAND_0000069 0.18
CAND_0000030 0.164
CAND_0000092 0.178


In [6]:
from explanation import ExplanationGenerator

In [9]:
explainer = ExplanationGenerator()

In [16]:
for i, result in enumerate(rankings[:5], start=1):

    candidate = result["candidate"]

    print("=" * 80)
    print(f"Rank {i}: {candidate.candidate_id}")

    print(f"Final      : {result['final_score']:.3f}")
    print(f"Evidence   : {result['evidence_score']:.3f}")
    print(f"Skills     : {result['skill_score']:.3f}")
    print(f"Experience : {result['experience_score']:.3f}")
    print(f"Behavior   : {result['behavior_score']:.3f}")

    print("\nDocument:")
    print(candidate.retrieval_document[:800])   # First 800 characters

    print("\nReasoning:")
    print(
        explainer.generate_reasoning(
            job,
            candidate,
            result
        )
    )
    print()

Rank 1: CAND_0000031
Final      : 0.478
Evidence   : 0.309
Skills     : 0.100
Experience : 1.000
Behavior   : 0.793

Document:
Summary: Machine learning engineer with 6.0 years of experience building ML-powered features in production. Strong background in NLP, recommendation systems, and applied AI; comfortable across the ML stack from feature engineering through deployment. Recently, I led the team that migrated our keyword-search-based product to embedding-based retrieval. I've learned that most retrieval problems are actually evaluation problems in disguise. My academic background is in CS/ML but my main learning has come from shipping real systems and seeing what holds up under production load. Open to senior IC roles in applied ML or AI engineering, ideally at product companies where I'd own a meaningful piece of the ML stack.
Headline: Recommendation Systems Engineer | Search, Ranking & Retrieval
Skills: Go,

Reasoning:
Production experience in recommendation systems. Key skills 